In [3]:
import ee

# 1️⃣ Authentification et initialisation
ee.Authenticate()  # à faire une seule fois, ouvre une page web


KeyboardInterrupt: Interrupted by user

In [4]:
ee.Initialize(project='RNprjt')  # ton projet Earth Engine

EEException: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.

In [ ]:
# Définir la zone (Arkansas ou Californie) et la période
area = ee.Geometry.Point([-92.3, 34.7]) # Exemple pour Arkansas
start_date = '2021-01-01'
end_date = '2021-12-31'

# Charger la collection Sentinel-2
s2_collection = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                 .filterBounds(area)
                 .filterDate(start_date, end_date)
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))) # Filtre les nuages

In [ ]:
# 4️⃣ Prétraitement : sélectionner bandes et normaliser
def preprocess_s2(image):
    bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
    return image.select(bands).divide(10000)

dataset = s2_collection.map(preprocess_s2)


In [ ]:
# 5️⃣ Charger les labels CDL
cdl = ee.Image("USDA/NASS/CDL/2021").select('cropland')

In [ ]:
# 6️⃣ Combiner Sentinel et labels pour échantillonnage
# On prend 10 000 pixels aléatoires
samples = dataset.mean().addBands(cdl).sample(
    region=area,
    scale=10,  # résolution en mètres
    numPixels=10000,
    seed=42,
    geometries=True
)

In [ ]:
# 7️⃣ Exporter les samples vers Google Drive
task = ee.batch.Export.table.toDrive(
    collection=samples,
    description='Sentinel2_CDL_samples',
    fileFormat='CSV'
)
task.start()

print("Export lancé ! Vérifie ton Google Drive.")